# Sentiment Analysis with Amazon Aurora PostgreSQL & AWS Comprehend ML Integration

## Overview

This notebook walks you through performing **sentiment analysis** on customer reviews by integrating **Amazon Aurora PostgreSQL** with **Amazon Comprehend** — all through Python code, without touching the AWS Console.

**Scenario:** You work at an e-commerce company that stores customer reviews in Aurora. By integrating Aurora with Amazon Comprehend, you can automatically analyze the sentiment of each review, quickly identify negative feedback, and improve customer satisfaction.

---

## What You Will Do

1. Create an S3 bucket and upload a sample customer reviews dataset
2. Create IAM roles granting Aurora access to Comprehend and S3
3. Create an Amazon Aurora PostgreSQL DB cluster
4. Connect to Aurora using `psycopg2` (Python PostgreSQL client)
5. Install AWS ML extensions and run sentiment analysis queries via Comprehend
6. Clean up all resources

---

## Prerequisites

- This notebook should run on an **Amazon SageMaker Notebook Instance** (or SageMaker Studio)
- The SageMaker execution role must have permissions for: **RDS, IAM, S3, EC2 (VPC), Comprehend**
- Estimated cost: **< $1** if completed within 1–2 hours
- Estimated time: **~60 minutes**

> ⚠️ **Important:** Run the **Cleanup** section at the end to avoid ongoing charges.

---
## Section 0: Install Dependencies

We install `psycopg2-binary` — the Python driver for PostgreSQL — which lets us connect to Aurora from this notebook, just like pgAdmin would from a desktop machine.

In [1]:
# Install the PostgreSQL Python driver
# psycopg2-binary is a self-contained package (no system dependencies needed)
!pip install psycopg2-binary --quiet

print("✅ Dependencies installed.")

✅ Dependencies installed.


---
## Section 1: Import Libraries & Configure AWS Session

We use `boto3` — the AWS SDK for Python — to interact with AWS services programmatically.
SageMaker automatically provides credentials through the notebook's execution role.

In [2]:
import boto3
import json
import time
import psycopg2
from botocore.exceptions import ClientError

# ─────────────────────────────────────────────
# AWS Session & Region
# SageMaker automatically uses the notebook's execution role for credentials
# ─────────────────────────────────────────────
session    = boto3.session.Session()
region     = session.region_name          # e.g. 'us-east-1'
account_id = boto3.client('sts').get_caller_identity()['Account']

print(f"✅ AWS Session initialized")
print(f"   Region     : {region}")
print(f"   Account ID : {account_id}")

# ─────────────────────────────────────────────
# Global resource names — change these if needed
# ─────────────────────────────────────────────
BUCKET_NAME           = f"aurora-reviews-lab-{account_id}"   # S3 bucket (must be globally unique)
TSV_FILE_NAME         = "sample_us.tsv"                       # Dataset file
COMPREHEND_ROLE_NAME  = "AuroraComprehendRole"                 # IAM role for Comprehend
S3_ROLE_NAME          = "AuroraS3Role"                        # IAM role for S3
DB_CLUSTER_ID         = "aurora-ml-db"                        # Aurora cluster identifier
DB_NAME               = "postgres"                            # Default DB name
DB_USER               = "postgres"                            # Master username
DB_PASSWORD           = "AuroraML2024!"                       # ⚠️ Change this to a strong password
VPC_NAME              = "aurora-ml-vpc"                       # VPC tag name

print("\n📋 Configuration:")
print(f"   S3 Bucket        : {BUCKET_NAME}")
print(f"   DB Cluster ID    : {DB_CLUSTER_ID}")
print(f"   DB User          : {DB_USER}")
print(f"   Comprehend Role  : {COMPREHEND_ROLE_NAME}")
print(f"   S3 Role          : {S3_ROLE_NAME}")

✅ AWS Session initialized
   Region     : us-east-1
   Account ID : 745404749079

📋 Configuration:
   S3 Bucket        : aurora-reviews-lab-745404749079
   DB Cluster ID    : aurora-ml-db
   DB User          : postgres
   Comprehend Role  : AuroraComprehendRole
   S3 Role          : AuroraS3Role


---
## Section 2: Create S3 Bucket & Upload Dataset

We create a private S3 bucket and upload the **sample_us.tsv** customer reviews file.
This file will later be imported directly into Aurora PostgreSQL using the `aws_s3` extension.

> 💡 The original lab downloaded this file manually; here we fetch it programmatically.

In [3]:
s3_client = boto3.client('s3', region_name=region)

# ── Step 2.1: Create the S3 Bucket ─────────────────────────────────────────
print(f"Creating S3 bucket: {BUCKET_NAME} in {region} ...")

try:
    if region == 'us-east-1':
        # us-east-1 does not accept a LocationConstraint
        s3_client.create_bucket(Bucket=BUCKET_NAME)
    else:
        s3_client.create_bucket(
            Bucket=BUCKET_NAME,
            CreateBucketConfiguration={'LocationConstraint': region}
        )
    print(f"✅ Bucket '{BUCKET_NAME}' created.")
except ClientError as e:
    if e.response['Error']['Code'] in ('BucketAlreadyOwnedByYou', 'BucketAlreadyExists'):
        print(f"ℹ️  Bucket '{BUCKET_NAME}' already exists — continuing.")
    else:
        raise

# ── Step 2.2: Block all public access (security best practice) ──────────────
s3_client.put_public_access_block(
    Bucket=BUCKET_NAME,
    PublicAccessBlockConfiguration={
        'BlockPublicAcls': True,
        'IgnorePublicAcls': True,
        'BlockPublicPolicy': True,
        'RestrictPublicBuckets': True
    }
)
print("✅ Public access blocked on bucket.")

Creating S3 bucket: aurora-reviews-lab-745404749079 in us-east-1 ...


✅ Bucket 'aurora-reviews-lab-745404749079' created.
✅ Public access blocked on bucket.


In [4]:
# ── Step 2.3: Generate & upload sample_us.tsv ───────────────────────────────
# The original lab downloads this file from the web.
# Here we create a realistic 15-row sample that matches the Amazon Reviews schema.

sample_tsv_content = """marketplace\tcustomer_id\treview_id\tproduct_id\tproduct_parent\tproduct_title\tproduct_category\tstar_rating\thelpful_votes\ttotal_votes\tvine\tverified_purchase\treview_headline\treview_body\treview_date
US\t18778586\tRZSLHH1NKO6OX\tB00BGGDVOO\t738692522\tAll-New Kindle E-reader\tElectronics\t5\t0\t0\tN\tY\tExcellent product\tThis Kindle is amazing! Best e-reader I have ever used. The display is crystal clear and battery lasts weeks.\t2015-08-31
US\t15322083\tR30SHQMEBHAJNO\tB00BGGDVOO\t738692522\tAll-New Kindle E-reader\tElectronics\t1\t0\t1\tN\tY\tDisappointing\tTotal waste of money. Stopped working after two days. Customer service was unhelpful and rude.\t2015-08-31
US\t20381037\tROWCLNS7AIPOY\tB00BGGDVOO\t738692522\tAll-New Kindle E-reader\tElectronics\t3\t1\t2\tN\tY\tOkay product\tIt works as expected, nothing special. The screen could be better and the UI feels outdated.\t2015-08-31
US\t24803302\tR36B0NM02Y1Z5A\tB00BGGDVOO\t738692522\tAll-New Kindle E-reader\tElectronics\t5\t3\t3\tN\tY\tLove it!\tAbsolutely love this Kindle. Reading books has never been more enjoyable. Highly recommend to everyone!\t2015-08-31
US\t11398059\tR3QVGDNMWO0HZ8\tB00BGGDVOO\t738692522\tAll-New Kindle E-reader\tElectronics\t2\t2\t3\tN\tN\tNot impressed\tI was really hoping for more. The device feels cheap and the software crashes frequently. Not worth the price.\t2015-08-31
US\t23688145\tR1KD0W7R3RVVLF\tB001GVISJM\t123456789\tWireless Bluetooth Headphones\tElectronics\t4\t5\t6\tN\tY\tGreat sound quality\tVery good headphones for the price. Sound is rich and bass is deep. Comfortable for long listening sessions.\t2015-09-01
US\t32145678\tR2WXYZ12345678\tB001GVISJM\t123456789\tWireless Bluetooth Headphones\tElectronics\t1\t0\t2\tN\tY\tBroke immediately\tThe headphones broke after just one week. The build quality is terrible. I am furious and want a refund.\t2015-09-01
US\t45678901\tR3ABCD98765432\tB002HELLOOO\t987654321\tPortable Charger 10000mAh\tElectronics\t5\t10\t11\tN\tY\tMust have!\tThis power bank is a lifesaver when traveling. Charges my phone three times fully. Compact and lightweight design.\t2015-09-02
US\t56789012\tR4EFGH11223344\tB002HELLOOO\t987654321\tPortable Charger 10000mAh\tElectronics\t3\t1\t4\tN\tY\tAverage\tDoes the job but nothing exceptional. Takes a long time to fully charge itself. Packaging was decent.\t2015-09-02
US\t67890123\tR5IJKL55667788\tB003WORLDDD\t111222333\tSmart LED Bulb\tHome\t5\t7\t7\tN\tY\tAmazing smart bulb\tThese bulbs transformed my home. Easy to set up with the app and the color range is fantastic. Love them!\t2015-09-03
US\t78901234\tR6MNOP99001122\tB003WORLDDD\t111222333\tSmart LED Bulb\tHome\t2\t0\t1\tN\tN\tDidn't work\tConnected once then never again. The app is buggy and support docs are useless. Returning these immediately.\t2015-09-03
US\t89012345\tR7QRST33445566\tB004TESTTT\t444555666\tCoffee Maker 12-Cup\tKitchen\t4\t4\t5\tN\tY\tGreat coffee maker\tMakes excellent coffee every morning. Easy to clean and the carafe keeps coffee hot for hours. Very happy.\t2015-09-04
US\t90123456\tR8UVWX77889900\tB004TESTTT\t444555666\tCoffee Maker 12-Cup\tKitchen\t1\t3\t5\tN\tY\tLeaks badly\tThe coffee maker leaks all over my counter every single time I use it. Completely useless appliance. Terrible quality.\t2015-09-04
US\t10234567\tR9YZAB12349876\tB005SAMPLEL\t777888999\tYoga Mat Non-Slip\tSports\t5\t2\t2\tN\tY\tPerfect yoga mat\tOutstanding quality mat. Non-slip surface works perfectly even during hot yoga. Great thickness and very durable.\t2015-09-05
US\t20345678\tR0CDEF56781234\tB005SAMPLEL\t777888999\tYoga Mat Non-Slip\tSports\t3\t0\t0\tN\tY\tDecent but not great\tThe mat is okay for basic yoga. The edges curl up after a few sessions which is a bit annoying during practice.\t2015-09-05
"""

# Write to a local file first
with open(TSV_FILE_NAME, 'w') as f:
    f.write(sample_tsv_content)

# Upload to S3
s3_client.upload_file(TSV_FILE_NAME, BUCKET_NAME, TSV_FILE_NAME)
print(f"✅ '{TSV_FILE_NAME}' uploaded to s3://{BUCKET_NAME}/{TSV_FILE_NAME}")
print(f"   Rows in file: {sample_tsv_content.count(chr(10)) - 1} data rows + 1 header")

✅ 'sample_us.tsv' uploaded to s3://aurora-reviews-lab-745404749079/sample_us.tsv
   Rows in file: 15 data rows + 1 header


---
## Section 3: Create IAM Roles

Aurora needs permission to call Comprehend and read from S3. We create two IAM roles:

| Role | Purpose | Policy Attached |
|---|---|---|
| `AuroraComprehendRole` | Allows Aurora to call Comprehend | `ComprehendReadOnly` |
| `AuroraS3Role` | Allows Aurora to read from S3 | `AmazonS3ReadOnlyAccess` |

Both roles use `rds.amazonaws.com` as the trusted service (equivalent to "RDS - Add Role to Database" in the console).

In [5]:
iam_client = boto3.client('iam')

# Trust policy: allows RDS service to assume these roles
rds_trust_policy = json.dumps({
    "Version": "2012-10-17",
    "Statement": [
        {
            "Effect": "Allow",
            "Principal": {"Service": "rds.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }
    ]
})

# ── Helper: create role if it doesn't already exist ──────────────────────────
def create_role_if_not_exists(role_name, trust_policy, managed_policy_arn, description):
    """Creates an IAM role and attaches a managed policy. Skips creation if role exists."""
    try:
        response = iam_client.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=trust_policy,
            Description=description
        )
        role_arn = response['Role']['Arn']
        print(f"   Created role: {role_name}")
    except ClientError as e:
        if e.response['Error']['Code'] == 'EntityAlreadyExists':
            role_arn = iam_client.get_role(RoleName=role_name)['Role']['Arn']
            print(f"   Role already exists: {role_name}")
        else:
            raise

    # Attach the managed policy
    try:
        iam_client.attach_role_policy(
            RoleName=role_name,
            PolicyArn=managed_policy_arn
        )
        print(f"   Attached policy: {managed_policy_arn.split('/')[-1]}")
    except ClientError as e:
        if 'already' in str(e).lower():
            print(f"   Policy already attached.")
        else:
            raise

    return role_arn

# ── Create AuroraComprehendRole ──────────────────────────────────────────────
print("\n📌 Creating AuroraComprehendRole ...")
comprehend_role_arn = create_role_if_not_exists(
    role_name          = COMPREHEND_ROLE_NAME,
    trust_policy       = rds_trust_policy,
    managed_policy_arn = "arn:aws:iam::aws:policy/ComprehendReadOnly",
    description        = "Allows Aurora PostgreSQL to call Amazon Comprehend for ML inference"
)
print(f"   ARN: {comprehend_role_arn}")

# ── Create AuroraS3Role ──────────────────────────────────────────────────────
print("\n📌 Creating AuroraS3Role ...")
s3_role_arn = create_role_if_not_exists(
    role_name          = S3_ROLE_NAME,
    trust_policy       = rds_trust_policy,
    managed_policy_arn = "arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess",
    description        = "Allows Aurora PostgreSQL to import data from Amazon S3"
)
print(f"   ARN: {s3_role_arn}")

print("\n✅ Both IAM roles are ready.")


📌 Creating AuroraComprehendRole ...
   Created role: AuroraComprehendRole
   Attached policy: ComprehendReadOnly
   ARN: arn:aws:iam::745404749079:role/AuroraComprehendRole

📌 Creating AuroraS3Role ...


   Created role: AuroraS3Role
   Attached policy: AmazonS3ReadOnlyAccess
   ARN: arn:aws:iam::745404749079:role/AuroraS3Role

✅ Both IAM roles are ready.


---
## Section 4: Create VPC & Security Group

Aurora must run inside a VPC. We create a new VPC with two subnets in different Availability Zones (required for Aurora DB subnet groups), and a Security Group that allows PostgreSQL traffic (port 5432) from this notebook's IP address.

> 💡 In the console lab you selected "Create new VPC" during DB creation. Here we do that explicitly so we have full control.

In [6]:
ec2_client   = boto3.client('ec2', region_name=region)
ec2_resource = boto3.resource('ec2', region_name=region)

# ── Step 4.1: Create VPC ─────────────────────────────────────────────────────
print("Creating VPC ...")
vpc_response = ec2_client.create_vpc(CidrBlock='10.0.0.0/16')
vpc_id = vpc_response['Vpc']['VpcId']

# Enable DNS support (required for Aurora)
ec2_client.modify_vpc_attribute(VpcId=vpc_id, EnableDnsSupport={'Value': True})
ec2_client.modify_vpc_attribute(VpcId=vpc_id, EnableDnsHostnames={'Value': True})

# Tag the VPC for easy identification
ec2_client.create_tags(Resources=[vpc_id], Tags=[{'Key': 'Name', 'Value': VPC_NAME}])
print(f"✅ VPC created: {vpc_id}")

# ── Step 4.2: Create Internet Gateway ───────────────────────────────────────
igw = ec2_client.create_internet_gateway()
igw_id = igw['InternetGateway']['InternetGatewayId']
ec2_client.attach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
print(f"✅ Internet Gateway created and attached: {igw_id}")

# ── Step 4.3: Create two public subnets in different AZs ────────────────────
# Get available AZs in this region
azs = [az['ZoneName'] for az in ec2_client.describe_availability_zones()['AvailabilityZones']
       if az['State'] == 'available'][:2]

subnet_ids = []
cidrs = ['10.0.1.0/24', '10.0.2.0/24']

for i, (az, cidr) in enumerate(zip(azs, cidrs)):
    subnet = ec2_client.create_subnet(VpcId=vpc_id, CidrBlock=cidr, AvailabilityZone=az)
    sid = subnet['Subnet']['SubnetId']
    # Enable public IPs so we can connect from SageMaker
    ec2_client.modify_subnet_attribute(SubnetId=sid, MapPublicIpOnLaunch={'Value': True})
    ec2_client.create_tags(Resources=[sid], Tags=[{'Key': 'Name', 'Value': f'aurora-ml-subnet-{i+1}'}])
    subnet_ids.append(sid)
    print(f"✅ Subnet {i+1} created: {sid} in {az}")

# ── Step 4.4: Create Route Table & add route to Internet Gateway ─────────────
rt = ec2_client.create_route_table(VpcId=vpc_id)
rt_id = rt['RouteTable']['RouteTableId']
ec2_client.create_route(RouteTableId=rt_id, DestinationCidrBlock='0.0.0.0/0', GatewayId=igw_id)
for sid in subnet_ids:
    ec2_client.associate_route_table(RouteTableId=rt_id, SubnetId=sid)
print(f"✅ Route table configured: {rt_id}")

Creating VPC ...


✅ VPC created: vpc-0d26514770cf63e2c


✅ Internet Gateway created and attached: igw-055ec5c16d92c6d29


✅ Subnet 1 created: subnet-05e276e45139dd082 in us-east-1a


✅ Subnet 2 created: subnet-08559313f3910a0a4 in us-east-1b


✅ Route table configured: rtb-0f5fa56b8304547e5


In [7]:
# ── Step 4.5: Create Security Group allowing PostgreSQL inbound ───────────────
# We allow port 5432 from 0.0.0.0/0 for lab purposes.
# In production, restrict this to your specific IP range.

sg = ec2_client.create_security_group(
    GroupName='aurora-ml-lab-sg',
    Description='Security group for Aurora ML lab - allows PostgreSQL',
    VpcId=vpc_id
)
sg_id = sg['GroupId']

# Allow inbound PostgreSQL from anywhere (suitable for this lab)
ec2_client.authorize_security_group_ingress(
    GroupId=sg_id,
    IpPermissions=[
        {
            'IpProtocol': 'tcp',
            'FromPort': 5432,
            'ToPort': 5432,
            'IpRanges': [{'CidrIp': '0.0.0.0/0', 'Description': 'PostgreSQL access for lab'}]
        }
    ]
)
print(f"✅ Security group created: {sg_id}")
print("   Port 5432 open for inbound connections")

print(f"""
📋 Network Summary:
   VPC ID        : {vpc_id}
   Subnet IDs    : {subnet_ids}
   Sec Group ID  : {sg_id}
""")

✅ Security group created: sg-09371411b1afac21f
   Port 5432 open for inbound connections

📋 Network Summary:
   VPC ID        : vpc-0d26514770cf63e2c
   Subnet IDs    : ['subnet-05e276e45139dd082', 'subnet-08559313f3910a0a4']
   Sec Group ID  : sg-09371411b1afac21f



---
## Section 5: Create Aurora PostgreSQL DB Cluster

We create an **Aurora PostgreSQL** cluster (compatible with PostgreSQL 15) with:
- Engine: `aurora-postgresql`
- Instance class: `db.t3.medium` (matches the console lab)
- Single writer instance (no Multi-AZ replica — same as console lab)
- Public accessibility enabled so we can connect from SageMaker

> ⏳ **This step takes 5–10 minutes.** The cell waits until the cluster is available.

In [8]:
rds_client = boto3.client('rds', region_name=region)

# ── Step 5.1: Create DB Subnet Group ─────────────────────────────────────────
# A DB subnet group tells Aurora which subnets it can use
print("Creating DB subnet group ...")
try:
    rds_client.create_db_subnet_group(
        DBSubnetGroupName='aurora-ml-subnet-group',
        DBSubnetGroupDescription='Subnet group for Aurora ML lab',
        SubnetIds=subnet_ids,
        Tags=[{'Key': 'Name', 'Value': 'aurora-ml-subnet-group'}]
    )
    print("✅ DB subnet group created.")
except ClientError as e:
    if 'already exists' in str(e).lower():
        print("ℹ️  DB subnet group already exists — continuing.")
    else:
        raise

Creating DB subnet group ...


✅ DB subnet group created.


In [9]:
# ── Step 5.1b: Pick the latest available Aurora PostgreSQL 15.x in this region ──
# Hardcoding a minor version (e.g. '15.4') breaks once AWS retires it, so we
# query what's actually available and use the newest 15.x.
_versions = rds_client.describe_db_engine_versions(Engine='aurora-postgresql')['DBEngineVersions']
_v15 = sorted(
    (v['EngineVersion'] for v in _versions if v['EngineVersion'].startswith('15')),
    key=lambda s: [int(x) for x in s.split('.')]
)
ENGINE_VERSION = _v15[-1] if _v15 else '15.4'
print(f"Using Aurora PostgreSQL engine version: {ENGINE_VERSION}")

# ── Step 5.2: Create Aurora Cluster ──────────────────────────────────────────
# This is the equivalent of clicking "Create database" in the RDS console

print(f"Creating Aurora cluster '{DB_CLUSTER_ID}' ...")
print("(This may take 5–10 minutes — please wait)\n")

try:
    cluster_response = rds_client.create_db_cluster(
        DBClusterIdentifier = DB_CLUSTER_ID,
        Engine              = 'aurora-postgresql',
        EngineVersion       = ENGINE_VERSION,   # latest available Aurora PostgreSQL 15.x
        DatabaseName        = DB_NAME,
        MasterUsername      = DB_USER,
        MasterUserPassword  = DB_PASSWORD,
        VpcSecurityGroupIds = [sg_id],
        DBSubnetGroupName   = 'aurora-ml-subnet-group',
        StorageEncrypted    = True,            # Best practice: encrypt at rest
        BackupRetentionPeriod = 1,
        Tags=[{'Key': 'Name', 'Value': 'aurora-ml-lab-cluster'}]
    )
    print("✅ Aurora cluster creation initiated.")

except ClientError as e:
    if 'already exists' in str(e).lower():
        print("ℹ️  Cluster already exists — continuing.")
    else:
        raise

# ── Step 5.3: Create DB Instance (the writer node) ────────────────────────────
print(f"\nCreating DB instance '{DB_CLUSTER_ID}-instance-1' ...")
try:
    rds_client.create_db_instance(
        DBInstanceIdentifier = f'{DB_CLUSTER_ID}-instance-1',
        DBClusterIdentifier  = DB_CLUSTER_ID,
        DBInstanceClass      = 'db.t3.medium',
        Engine               = 'aurora-postgresql',
        PubliclyAccessible   = True,           # Needed to connect from SageMaker
        EnablePerformanceInsights = False,     # Off to match lab config
        Tags=[{'Key': 'Name', 'Value': 'aurora-ml-lab-instance'}]
    )
    print("✅ DB instance creation initiated.")

except ClientError as e:
    if 'already exists' in str(e).lower():
        print("ℹ️  DB instance already exists — continuing.")
    else:
        raise

Using Aurora PostgreSQL engine version: 15.17
Creating Aurora cluster 'aurora-ml-db' ...
(This may take 5–10 minutes — please wait)



✅ Aurora cluster creation initiated.

Creating DB instance 'aurora-ml-db-instance-1' ...


✅ DB instance creation initiated.


In [10]:
# ── Step 5.4: Wait for the cluster and instance to become available ───────────
# This polls every 30 seconds until the DB is ready (typically 5–10 minutes)

print("⏳ Waiting for Aurora cluster to become available ...")
print("   (Polling every 30 seconds — please be patient)\n")

waiter = rds_client.get_waiter('db_instance_available')
waiter.wait(
    DBInstanceIdentifier=f'{DB_CLUSTER_ID}-instance-1',
    WaiterConfig={'Delay': 30, 'MaxAttempts': 40}  # max 20 minutes
)

# ── Step 5.5: Retrieve the cluster endpoint ────────────────────────────────────
cluster_info = rds_client.describe_db_clusters(DBClusterIdentifier=DB_CLUSTER_ID)
cluster      = cluster_info['DBClusters'][0]
DB_ENDPOINT  = cluster['Endpoint']     # Writer endpoint
DB_PORT      = cluster['Port']         # Default: 5432

print(f"\n✅ Aurora cluster is AVAILABLE!")
print(f"   Endpoint : {DB_ENDPOINT}")
print(f"   Port     : {DB_PORT}")
print(f"   Status   : {cluster['Status']}")

⏳ Waiting for Aurora cluster to become available ...
   (Polling every 30 seconds — please be patient)




✅ Aurora cluster is AVAILABLE!
   Endpoint : aurora-ml-db.cluster-cy1k44844jip.us-east-1.rds.amazonaws.com
   Port     : 5432
   Status   : available


---
## Section 6: Attach IAM Roles to the Aurora Cluster

Now we associate the two IAM roles with the Aurora cluster:
- `AuroraComprehendRole` → feature: `Comprehend`
- `AuroraS3Role` → feature: `s3Import`

This is equivalent to the **"Manage IAM roles"** step in the console's Connectivity & Security tab.

> ⏳ After associating roles, wait a few minutes for them to become `ACTIVE`.

In [11]:
# ── Attach AuroraComprehendRole ──────────────────────────────────────────────
print("Attaching AuroraComprehendRole to cluster ...")
try:
    rds_client.add_role_to_db_cluster(
        DBClusterIdentifier = DB_CLUSTER_ID,
        RoleArn             = comprehend_role_arn,
        FeatureName         = 'Comprehend'
    )
    print("   ✅ Comprehend role association initiated.")
except ClientError as e:
    print(f"   ℹ️  {e.response['Error']['Message']}")

# ── Attach AuroraS3Role ──────────────────────────────────────────────────────
print("Attaching AuroraS3Role to cluster ...")
try:
    rds_client.add_role_to_db_cluster(
        DBClusterIdentifier = DB_CLUSTER_ID,
        RoleArn             = s3_role_arn,
        FeatureName         = 's3Import'
    )
    print("   ✅ S3 role association initiated.")
except ClientError as e:
    print(f"   ℹ️  {e.response['Error']['Message']}")

# ── Wait for roles to become ACTIVE ─────────────────────────────────────────
print("\n⏳ Waiting for IAM roles to become ACTIVE (up to 5 minutes) ...")

for attempt in range(20):
    time.sleep(30)
    cluster_details = rds_client.describe_db_clusters(DBClusterIdentifier=DB_CLUSTER_ID)
    roles = cluster_details['DBClusters'][0].get('AssociatedRoles', [])
    active_roles = [r for r in roles if r['Status'] == 'ACTIVE']
    pending_roles = [r for r in roles if r['Status'] != 'ACTIVE']

    print(f"   Attempt {attempt+1}: {len(active_roles)} ACTIVE, {len(pending_roles)} pending")

    if len(active_roles) >= 2:
        print("\n✅ Both IAM roles are now ACTIVE!")
        for r in active_roles:
            print(f"   - {r['RoleArn'].split('/')[-1]} → {r['Status']}")
        break
else:
    print("\n⚠️  Roles are taking longer than expected. You may proceed; they should activate shortly.")

Attaching AuroraComprehendRole to cluster ...
   ✅ Comprehend role association initiated.
Attaching AuroraS3Role to cluster ...


   ✅ S3 role association initiated.

⏳ Waiting for IAM roles to become ACTIVE (up to 5 minutes) ...


   Attempt 1: 2 ACTIVE, 0 pending

✅ Both IAM roles are now ACTIVE!
   - AuroraComprehendRole → ACTIVE
   - AuroraS3Role → ACTIVE


---
## Section 7: Connect to Aurora PostgreSQL

We use `psycopg2` to open a connection to Aurora — this is the Python equivalent of using pgAdmin or any other PostgreSQL client.

> 💡 If connection fails, check that the Security Group allows port 5432 and that the cluster is publicly accessible.

In [12]:
# ── Connect to Aurora ────────────────────────────────────────────────────────
print(f"Connecting to Aurora at {DB_ENDPOINT}:{DB_PORT} ...")

conn = psycopg2.connect(
    host     = DB_ENDPOINT,
    port     = DB_PORT,
    dbname   = DB_NAME,
    user     = DB_USER,
    password = DB_PASSWORD,
    connect_timeout = 30
)
conn.autocommit = True   # Commit each statement automatically
cursor = conn.cursor()

print("✅ Connected to Aurora PostgreSQL!")

# Quick verification
cursor.execute("SELECT version();")
db_version = cursor.fetchone()[0]
print(f"   Server version: {db_version[:60]}...")

# ── Helper: run a query and print results in a readable table ─────────────────
def run_query(sql, show_results=True, label=None):
    """Execute SQL and optionally display the results."""
    if label:
        print(f"\n{'─'*60}")
        print(f"🔍 {label}")
        print('─'*60)
    print(f"SQL: {sql[:120]}..." if len(sql) > 120 else f"SQL: {sql}")
    cursor.execute(sql)
    if show_results and cursor.description:
        cols = [desc[0] for desc in cursor.description]
        rows = cursor.fetchall()
        # Print header
        col_widths = [max(len(str(col)), max((len(str(r[i])) for r in rows), default=0)) 
                      for i, col in enumerate(cols)]
        header = ' | '.join(str(c).ljust(w) for c, w in zip(cols, col_widths))
        print(f"\n{header}")
        print('─' * len(header))
        for row in rows:
            print(' | '.join(str(v).ljust(w) for v, w in zip(row, col_widths)))
        print(f"\n({len(rows)} row{'s' if len(rows) != 1 else ''} returned)")
        return rows
    else:
        print("✅ Statement executed successfully.")
        return None

Connecting to Aurora at aurora-ml-db.cluster-cy1k44844jip.us-east-1.rds.amazonaws.com:5432 ...
✅ Connected to Aurora PostgreSQL!
   Server version: PostgreSQL 15.17 on x86_64-pc-linux-gnu, compiled by x86_64-...


---
## Section 8: Install AWS ML Extension

The `aws_ml` extension unlocks the `aws_comprehend` schema inside PostgreSQL — providing the `detect_sentiment()` function that calls Amazon Comprehend directly from SQL.

```sql
CREATE EXTENSION IF NOT EXISTS aws_ml CASCADE;
```

The `CASCADE` keyword also installs any dependent extensions automatically.

In [13]:
# Install the AWS ML extension — enables aws_comprehend.detect_sentiment()
run_query(
    "CREATE EXTENSION IF NOT EXISTS aws_ml CASCADE;",
    show_results=False,
    label="Installing AWS ML Extension"
)

# Verify installed extensions
run_query(
    "SELECT extname, extversion FROM pg_extension WHERE extname LIKE 'aws%';",
    label="Verifying AWS extensions installed"
)


────────────────────────────────────────────────────────────
🔍 Installing AWS ML Extension
────────────────────────────────────────────────────────────
SQL: CREATE EXTENSION IF NOT EXISTS aws_ml CASCADE;
✅ Statement executed successfully.

────────────────────────────────────────────────────────────
🔍 Verifying AWS extensions installed
────────────────────────────────────────────────────────────
SQL: SELECT extname, extversion FROM pg_extension WHERE extname LIKE 'aws%';

extname     | extversion
────────────────────────
aws_commons | 1.2       
aws_ml      | 2.0       

(2 rows returned)


[('aws_commons', '1.2'), ('aws_ml', '2.0')]

---
## Section 9: Create Sample Table & Run Sentiment Analysis

We now reproduce the exact queries from the original lab:

1. Create a `comments` table
2. Insert sample comments
3. Call `aws_comprehend.detect_sentiment()` on those comments

Comprehend returns two values per row:
- **`sentiment`** — `POSITIVE`, `NEGATIVE`, `NEUTRAL`, or `MIXED`
- **`confidence`** — a score from 0 to 1 indicating model certainty

In [14]:
# ── Step 9.1: Create the comments table ──────────────────────────────────────
run_query(
    """
    CREATE TABLE IF NOT EXISTS comments (
        comment_id   serial PRIMARY KEY,
        comment_text VARCHAR(255) NOT NULL
    );
    """,
    show_results=False,
    label="Create 'comments' table"
)

# ── Step 9.2: Insert sample comment rows ─────────────────────────────────────
# These are the exact four comments from the original lab guide
comments = [
    'This is very useful, thank you for writing it!',
    'Awesome, I was waiting for this feature.',
    'An interesting write up, please add more details.',
    'I do not like how this was implemented.'
]

# Clear existing rows so re-running this cell doesn't duplicate data
cursor.execute("TRUNCATE TABLE comments RESTART IDENTITY;")

for comment in comments:
    cursor.execute("INSERT INTO comments (comment_text) VALUES (%s);", (comment,))

print(f"✅ Inserted {len(comments)} comments into the table.")

# Verify the inserts
run_query("SELECT * FROM comments;", label="Verify inserted comments")


────────────────────────────────────────────────────────────
🔍 Create 'comments' table
────────────────────────────────────────────────────────────
SQL: 
    CREATE TABLE IF NOT EXISTS comments (
        comment_id   serial PRIMARY KEY,
        comment_text VARCHAR(255) NO...
✅ Statement executed successfully.
✅ Inserted 4 comments into the table.

────────────────────────────────────────────────────────────
🔍 Verify inserted comments
────────────────────────────────────────────────────────────
SQL: SELECT * FROM comments;

comment_id | comment_text                                     
──────────────────────────────────────────────────────────────
1          | This is very useful, thank you for writing it!   
2          | Awesome, I was waiting for this feature.         
3          | An interesting write up, please add more details.
4          | I do not like how this was implemented.          

(4 rows returned)


[(1, 'This is very useful, thank you for writing it!'),
 (2, 'Awesome, I was waiting for this feature.'),
 (3, 'An interesting write up, please add more details.'),
 (4, 'I do not like how this was implemented.')]

In [15]:
# ── Step 9.3: Call aws_comprehend.detect_sentiment on the comments ────────────
# This is the key query from the lab — it passes each comment to Comprehend
# and returns the detected sentiment + confidence score alongside the original data.

run_query(
    """
    SELECT 
        comment_id,
        comment_text,
        s.sentiment,
        ROUND(s.confidence::numeric, 4) AS confidence
    FROM comments,
         aws_comprehend.detect_sentiment(comments.comment_text, 'en') AS s;
    """,
    label="Sentiment Analysis on comments table (via Amazon Comprehend)"
)


────────────────────────────────────────────────────────────
🔍 Sentiment Analysis on comments table (via Amazon Comprehend)
────────────────────────────────────────────────────────────
SQL: 
    SELECT 
        comment_id,
        comment_text,
        s.sentiment,
        ROUND(s.confidence::numeric, 4) AS c...

comment_id | comment_text                                      | sentiment | confidence
───────────────────────────────────────────────────────────────────────────────────────
1          | This is very useful, thank you for writing it!    | POSITIVE  | 0.9996    
2          | Awesome, I was waiting for this feature.          | POSITIVE  | 0.9977    
3          | An interesting write up, please add more details. | POSITIVE  | 0.8926    
4          | I do not like how this was implemented.           | NEGATIVE  | 0.9971    

(4 rows returned)


[(1,
  'This is very useful, thank you for writing it!',
  'POSITIVE',
  Decimal('0.9996')),
 (2,
  'Awesome, I was waiting for this feature.',
  'POSITIVE',
  Decimal('0.9977')),
 (3,
  'An interesting write up, please add more details.',
  'POSITIVE',
  Decimal('0.8926')),
 (4, 'I do not like how this was implemented.', 'NEGATIVE', Decimal('0.9971'))]

---
## Section 10: Install S3 Extension & Load Customer Reviews Dataset

Next we install the `aws_s3` extension, which lets Aurora import data directly from S3 using SQL. Then we create the `review_simple` table that mirrors the Amazon customer reviews schema, including two extra columns for sentiment results.

In [16]:
# ── Step 10.1: Install aws_s3 extension ──────────────────────────────────────
run_query(
    "CREATE EXTENSION IF NOT EXISTS aws_s3 CASCADE;",
    show_results=False,
    label="Installing AWS S3 Extension"
)

# Confirm both AWS extensions are installed
run_query(
    "SELECT extname, extversion FROM pg_extension WHERE extname LIKE 'aws%';",
    label="Verify all AWS extensions"
)


────────────────────────────────────────────────────────────
🔍 Installing AWS S3 Extension
────────────────────────────────────────────────────────────
SQL: CREATE EXTENSION IF NOT EXISTS aws_s3 CASCADE;
✅ Statement executed successfully.

────────────────────────────────────────────────────────────
🔍 Verify all AWS extensions
────────────────────────────────────────────────────────────
SQL: SELECT extname, extversion FROM pg_extension WHERE extname LIKE 'aws%';

extname     | extversion
────────────────────────
aws_commons | 1.2       
aws_ml      | 2.0       
aws_s3      | 2.0       

(3 rows returned)


[('aws_commons', '1.2'), ('aws_ml', '2.0'), ('aws_s3', '2.0')]

In [17]:
# ── Step 10.2: Create review_simple table ────────────────────────────────────
# Schema matches the Amazon Customer Reviews TSV dataset columns,
# plus scored_sentiment and scored_confidence for Comprehend output

run_query(
    """
    DROP TABLE IF EXISTS review_simple;

    CREATE TABLE review_simple (
        marketplace        char(2),
        customer_id        varchar(20),
        review_id          varchar(20)  PRIMARY KEY,
        product_id         varchar(20),
        product_parent     varchar(20),
        product_title      text,
        product_category   varchar(20),
        star_rating        int,
        helpful_votes      int,
        total_votes        int,
        vine               char,
        verified_purchase  char,
        review_headline    varchar(255),
        review_body        text,
        review_date        date,
        scored_sentiment   varchar(20),   -- Will be filled by Comprehend
        scored_confidence  float          -- Will be filled by Comprehend
    );
    """,
    show_results=False,
    label="Create review_simple table"
)

print("✅ review_simple table created.")


────────────────────────────────────────────────────────────
🔍 Create review_simple table
────────────────────────────────────────────────────────────
SQL: 
    DROP TABLE IF EXISTS review_simple;

    CREATE TABLE review_simple (
        marketplace        char(2),
        c...
✅ Statement executed successfully.
✅ review_simple table created.


In [18]:
# ── Step 10.3: Load data from S3 into Aurora ──────────────────────────────────
# aws_s3.table_import_from_s3 reads the TSV from your S3 bucket
# and inserts the rows into review_simple.
# Note: scored_sentiment and scored_confidence are not in the TSV — they stay NULL for now.

import_sql = f"""
SELECT aws_s3.table_import_from_s3(
    'review_simple',
    'marketplace, customer_id, review_id, product_id, product_parent, product_title,
     product_category, star_rating, helpful_votes, total_votes, vine, verified_purchase,
     review_headline, review_body, review_date',
    '(FORMAT CSV, HEADER true, DELIMITER E''\t'', QUOTE ''|'')',
    '{BUCKET_NAME}',
    '{TSV_FILE_NAME}',
    '{region}'
);
"""

print(f"Loading data from s3://{BUCKET_NAME}/{TSV_FILE_NAME} ...")
result = run_query(import_sql, label="Import TSV from S3 into Aurora")

# Verify row count
run_query(
    "SELECT COUNT(*) AS total_rows_loaded FROM review_simple;",
    label="Verify row count after import"
)

Loading data from s3://aurora-reviews-lab-745404749079/sample_us.tsv ...

────────────────────────────────────────────────────────────
🔍 Import TSV from S3 into Aurora
────────────────────────────────────────────────────────────
SQL: 
SELECT aws_s3.table_import_from_s3(
    'review_simple',
    'marketplace, customer_id, review_id, product_id, product_...

table_import_from_s3                                                                
────────────────────────────────────────────────────────────────────────────────────
15 rows imported into relation "review_simple" from file sample_us.tsv of 3555 bytes

(1 row returned)

────────────────────────────────────────────────────────────
🔍 Verify row count after import
────────────────────────────────────────────────────────────
SQL: SELECT COUNT(*) AS total_rows_loaded FROM review_simple;

total_rows_loaded
─────────────────
15               

(1 row returned)


[(15,)]

---
## Section 11: Run Comprehend Sentiment Analysis on Customer Reviews

Now the central part of the lab — we call Comprehend on each `review_body` and store the results back in the `scored_sentiment` and `scored_confidence` columns.

This approach is **cost-efficient**: we run inference once and cache results in the DB. Future queries read from the database, not Comprehend.

In [19]:
# ── Step 11.1: Call Comprehend and update the table ───────────────────────────
# For each row where scored_sentiment is NULL, Comprehend analyzes review_body
# and the UPDATE statement stores the result in the table.

run_query(
    """
    UPDATE review_simple
    SET 
        scored_sentiment  = s.sentiment,
        scored_confidence = s.confidence
    FROM review_simple AS src,
         aws_comprehend.detect_sentiment(src.review_body, 'en') AS s
    WHERE src.review_id = review_simple.review_id
      AND src.scored_sentiment IS NULL;
    """,
    show_results=False,
    label="Update review_simple with Comprehend sentiment scores"
)

print("✅ Sentiment analysis complete! Scores stored in the database.")


────────────────────────────────────────────────────────────
🔍 Update review_simple with Comprehend sentiment scores
────────────────────────────────────────────────────────────
SQL: 
    UPDATE review_simple
    SET 
        scored_sentiment  = s.sentiment,
        scored_confidence = s.confidence
   ...
✅ Statement executed successfully.
✅ Sentiment analysis complete! Scores stored in the database.


In [20]:
# ── Step 11.2: View all reviews with their sentiment scores ───────────────────
# This is Query 1 from the original lab

run_query(
    """
    SELECT 
        customer_id,
        review_id,
        LEFT(review_body, 60) || '...' AS review_body_preview,
        scored_sentiment,
        ROUND(scored_confidence::numeric, 4) AS scored_confidence
    FROM review_simple
    ORDER BY scored_sentiment, scored_confidence DESC;
    """,
    label="Query 1: All reviews with sentiment and confidence"
)


────────────────────────────────────────────────────────────
🔍 Query 1: All reviews with sentiment and confidence
────────────────────────────────────────────────────────────
SQL: 
    SELECT 
        customer_id,
        review_id,
        LEFT(review_body, 60) || '...' AS review_body_preview,
    ...

customer_id | review_id      | review_body_preview                                             | scored_sentiment | scored_confidence
─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
20345678    | R0CDEF56781234 | The mat is okay for basic yoga. The edges curl up after a fe... | MIXED            | 0.9850           
56789012    | R4EFGH11223344 | Does the job but nothing exceptional. Takes a long time to f... | MIXED            | 0.9833           
20381037    | ROWCLNS7AIPOY  | It works as expected, nothing special. The screen could be b... | MIXED            | 0.8931           
90123456    | R8UVWX7788

[('20345678',
  'R0CDEF56781234',
  'The mat is okay for basic yoga. The edges curl up after a fe...',
  'MIXED',
  Decimal('0.9850')),
 ('56789012',
  'R4EFGH11223344',
  'Does the job but nothing exceptional. Takes a long time to f...',
  'MIXED',
  Decimal('0.9833')),
 ('20381037',
  'ROWCLNS7AIPOY',
  'It works as expected, nothing special. The screen could be b...',
  'MIXED',
  Decimal('0.8931')),
 ('90123456',
  'R8UVWX77889900',
  'The coffee maker leaks all over my counter every single time...',
  'NEGATIVE',
  Decimal('0.9999')),
 ('15322083',
  'R30SHQMEBHAJNO',
  'Total waste of money. Stopped working after two days. Custom...',
  'NEGATIVE',
  Decimal('0.9999')),
 ('11398059',
  'R3QVGDNMWO0HZ8',
  'I was really hoping for more. The device feels cheap and the...',
  'NEGATIVE',
  Decimal('0.9998')),
 ('32145678',
  'R2WXYZ12345678',
  'The headphones broke after just one week. The build quality ...',
  'NEGATIVE',
  Decimal('0.9997')),
 ('78901234',
  'R6MNOP99001122',
  '

In [21]:
# ── Step 11.3: Summarize reviews grouped by sentiment ─────────────────────────
# This is Query 2 from the original lab — gives an at-a-glance distribution

run_query(
    """
    SELECT 
        scored_sentiment,
        COUNT(*) AS nReviews
    FROM review_simple
    GROUP BY scored_sentiment
    ORDER BY nReviews DESC;
    """,
    label="Query 2: Review count by sentiment category"
)


────────────────────────────────────────────────────────────
🔍 Query 2: Review count by sentiment category
────────────────────────────────────────────────────────────
SQL: 
    SELECT 
        scored_sentiment,
        COUNT(*) AS nReviews
    FROM review_simple
    GROUP BY scored_sentiment...

scored_sentiment | nreviews
───────────────────────────
POSITIVE         | 7       
NEGATIVE         | 5       
MIXED            | 3       

(3 rows returned)


[('POSITIVE', 7), ('NEGATIVE', 5), ('MIXED', 3)]

In [22]:
# ── Step 11.4: Filter to high-confidence predictions (> 0.9) ──────────────────
# This is Query 3 from the original lab — useful for acting only on clear signals

run_query(
    """
    SELECT 
        scored_sentiment,
        COUNT(*) AS nReviews
    FROM review_simple
    WHERE scored_confidence > 0.9
    GROUP BY scored_sentiment
    ORDER BY nReviews DESC;
    """,
    label="Query 3: High-confidence reviews only (confidence > 0.9)"
)


────────────────────────────────────────────────────────────
🔍 Query 3: High-confidence reviews only (confidence > 0.9)
────────────────────────────────────────────────────────────
SQL: 
    SELECT 
        scored_sentiment,
        COUNT(*) AS nReviews
    FROM review_simple
    WHERE scored_confidence >...

scored_sentiment | nreviews
───────────────────────────
POSITIVE         | 7       
NEGATIVE         | 5       
MIXED            | 2       

(3 rows returned)


[('POSITIVE', 7), ('NEGATIVE', 5), ('MIXED', 2)]

In [23]:
# ── Bonus Query: Star rating vs Comprehend sentiment comparison ───────────────
# This extra query shows how well star ratings align with ML-detected sentiment —
# a great insight for the e-commerce scenario!

run_query(
    """
    SELECT 
        star_rating,
        scored_sentiment,
        COUNT(*) AS review_count,
        ROUND(AVG(scored_confidence)::numeric, 3) AS avg_confidence
    FROM review_simple
    GROUP BY star_rating, scored_sentiment
    ORDER BY star_rating DESC, review_count DESC;
    """,
    label="Bonus: Star rating vs Comprehend sentiment alignment"
)

print("""
🎉 Congratulations! You have successfully:
   ✅ Created an Aurora PostgreSQL database
   ✅ Enabled integration with Amazon Comprehend
   ✅ Used Comprehend to perform sentiment analysis on customer reviews
   ✅ Stored sentiment results back in Aurora for efficient future queries
""")


────────────────────────────────────────────────────────────
🔍 Bonus: Star rating vs Comprehend sentiment alignment
────────────────────────────────────────────────────────────
SQL: 
    SELECT 
        star_rating,
        scored_sentiment,
        COUNT(*) AS review_count,
        ROUND(AVG(scored_c...

star_rating | scored_sentiment | review_count | avg_confidence
──────────────────────────────────────────────────────────────
5           | POSITIVE         | 5            | 0.999         
4           | POSITIVE         | 2            | 1.000         
3           | MIXED            | 3            | 0.954         
2           | NEGATIVE         | 2            | 1.000         
1           | NEGATIVE         | 3            | 1.000         

(5 rows returned)

🎉 Congratulations! You have successfully:
   ✅ Created an Aurora PostgreSQL database
   ✅ Enabled integration with Amazon Comprehend
   ✅ Used Comprehend to perform sentiment analysis on customer reviews
   ✅ Stored sentiment resul

---
## Section 12: Close the Database Connection

Always close the database connection when you're done to release resources.

In [24]:
cursor.close()
conn.close()
print("✅ Database connection closed.")

✅ Database connection closed.


---
## Section 13: Cleanup Resources

> ⚠️ **Run this section when you are completely finished with the lab.**
> Leaving Aurora running incurs charges (~$0.072/hour for db.t3.medium).

We delete resources in the correct dependency order:
1. Aurora DB Instance → Cluster
2. DB Subnet Group
3. Security Group → Route Table → Subnets → Internet Gateway → VPC
4. IAM Role policy detachments → Roles
5. S3 Bucket objects → Bucket

In [25]:
# ─────────────────────────────────────────────────────────────────────────────
# ⚠️  CLEANUP — Run this cell ONLY when you are done with the lab
# ─────────────────────────────────────────────────────────────────────────────

CONFIRM_CLEANUP = True   # Set to True when ready to delete all resources

if not CONFIRM_CLEANUP:
    print("Cleanup skipped. Set CONFIRM_CLEANUP = True to delete resources.")
else:
    print("🚨 Starting cleanup of all lab resources ...\n")

    # ── 1. Delete Aurora DB Instance ─────────────────────────────────────────
    try:
        print("Deleting Aurora DB instance ...")
        rds_client.delete_db_instance(
            DBInstanceIdentifier=f'{DB_CLUSTER_ID}-instance-1',
            SkipFinalSnapshot=True
        )
        print("   ✅ Delete initiated. Waiting for instance to be removed ...")
        waiter = rds_client.get_waiter('db_instance_deleted')
        waiter.wait(
            DBInstanceIdentifier=f'{DB_CLUSTER_ID}-instance-1',
            WaiterConfig={'Delay': 30, 'MaxAttempts': 40}
        )
        print("   ✅ DB instance deleted.")
    except ClientError as e:
        print(f"   ℹ️  {e.response['Error']['Message']}")

    # ── 2. Delete Aurora DB Cluster ──────────────────────────────────────────
    try:
        print("\nDeleting Aurora DB cluster ...")
        rds_client.delete_db_cluster(
            DBClusterIdentifier=DB_CLUSTER_ID,
            SkipFinalSnapshot=True
        )
        # Poll until deleted
        for _ in range(20):
            time.sleep(30)
            try:
                rds_client.describe_db_clusters(DBClusterIdentifier=DB_CLUSTER_ID)
                print("   Still deleting cluster ...")
            except ClientError:
                print("   ✅ DB cluster deleted.")
                break
    except ClientError as e:
        print(f"   ℹ️  {e.response['Error']['Message']}")

    # ── 3. Delete DB Subnet Group ─────────────────────────────────────────────
    try:
        print("\nDeleting DB subnet group ...")
        rds_client.delete_db_subnet_group(DBSubnetGroupName='aurora-ml-subnet-group')
        print("   ✅ DB subnet group deleted.")
    except ClientError as e:
        print(f"   ℹ️  {e.response['Error']['Message']}")

    # ── 4. Detach policies and delete IAM roles ───────────────────────────────
    print("\nDeleting IAM roles ...")
    for role_name, policy_arn in [
        (COMPREHEND_ROLE_NAME, 'arn:aws:iam::aws:policy/ComprehendReadOnly'),
        (S3_ROLE_NAME,         'arn:aws:iam::aws:policy/AmazonS3ReadOnlyAccess')
    ]:
        try:
            iam_client.detach_role_policy(RoleName=role_name, PolicyArn=policy_arn)
            iam_client.delete_role(RoleName=role_name)
            print(f"   ✅ Role '{role_name}' deleted.")
        except ClientError as e:
            print(f"   ℹ️  {role_name}: {e.response['Error']['Message']}")

    print("\n⏳ Pausing 15 seconds before VPC cleanup to allow dependencies to clear ...")
    time.sleep(15)

    # ── 5. Delete Security Group ──────────────────────────────────────────────
    try:
        print("\nDeleting security group ...")
        ec2_client.delete_security_group(GroupId=sg_id)
        print("   ✅ Security group deleted.")
    except ClientError as e:
        print(f"   ℹ️  {e.response['Error']['Message']}")

    # ── 6. Delete Subnets ─────────────────────────────────────────────────────
    print("\nDeleting subnets ...")
    for sid in subnet_ids:
        try:
            ec2_client.delete_subnet(SubnetId=sid)
            print(f"   ✅ Subnet {sid} deleted.")
        except ClientError as e:
            print(f"   ℹ️  {sid}: {e.response['Error']['Message']}")

    # ── 7. Delete Route Table ─────────────────────────────────────────────────
    try:
        print("\nDeleting route table ...")
        ec2_client.delete_route_table(RouteTableId=rt_id)
        print("   ✅ Route table deleted.")
    except ClientError as e:
        print(f"   ℹ️  {e.response['Error']['Message']}")

    # ── 8. Detach & Delete Internet Gateway ───────────────────────────────────
    try:
        print("\nDetaching and deleting Internet Gateway ...")
        ec2_client.detach_internet_gateway(InternetGatewayId=igw_id, VpcId=vpc_id)
        ec2_client.delete_internet_gateway(InternetGatewayId=igw_id)
        print("   ✅ Internet Gateway deleted.")
    except ClientError as e:
        print(f"   ℹ️  {e.response['Error']['Message']}")

    # ── 9. Delete VPC ─────────────────────────────────────────────────────────
    try:
        print("\nDeleting VPC ...")
        ec2_client.delete_vpc(VpcId=vpc_id)
        print(f"   ✅ VPC {vpc_id} deleted.")
    except ClientError as e:
        print(f"   ℹ️  {e.response['Error']['Message']}")

    # ── 10. Empty and Delete S3 Bucket ────────────────────────────────────────
    try:
        print(f"\nEmptying and deleting S3 bucket '{BUCKET_NAME}' ...")
        # Delete all objects first
        paginator = s3_client.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=BUCKET_NAME):
            for obj in page.get('Contents', []):
                s3_client.delete_object(Bucket=BUCKET_NAME, Key=obj['Key'])
                print(f"   Deleted object: {obj['Key']}")
        s3_client.delete_bucket(Bucket=BUCKET_NAME)
        print(f"   ✅ S3 bucket '{BUCKET_NAME}' deleted.")
    except ClientError as e:
        print(f"   ℹ️  {e.response['Error']['Message']}")

    print("""
╔══════════════════════════════════════════════════╗
║  ✅ All lab resources have been cleaned up!      ║
║     No ongoing charges will be incurred.         ║
╚══════════════════════════════════════════════════╝
""")

🚨 Starting cleanup of all lab resources ...

Deleting Aurora DB instance ...


   ✅ Delete initiated. Waiting for instance to be removed ...


   ✅ DB instance deleted.

Deleting Aurora DB cluster ...


   Still deleting cluster ...


   ✅ DB cluster deleted.

Deleting DB subnet group ...
   ✅ DB subnet group deleted.

Deleting IAM roles ...


...

   ✅ Subnet subnet-08559313f3910a0a4 deleted.

Deleting route table ...


   ✅ Route table deleted.

Detaching and deleting Internet Gateway ...


   ✅ Internet Gateway deleted.

Deleting VPC ...


   ✅ VPC vpc-0d26514770cf63e2c deleted.

Emptying and deleting S3 bucket 'aurora-reviews-lab-745404749079' ...
   Deleted object: sample_us.tsv


   ✅ S3 bucket 'aurora-reviews-lab-745404749079' deleted.

╔══════════════════════════════════════════════════╗
║  ✅ All lab resources have been cleaned up!      ║
║     No ongoing charges will be incurred.         ║
╚══════════════════════════════════════════════════╝



---
## Summary

In this lab you successfully:

| Step | What You Did |
|------|-------------|
| ✅ S3 Bucket | Created a bucket and uploaded customer review data |
| ✅ IAM Roles | Created `AuroraComprehendRole` and `AuroraS3Role` programmatically |
| ✅ Aurora Cluster | Created an Aurora PostgreSQL 15 cluster with a `db.t3.medium` writer |
| ✅ Role Attachment | Associated IAM roles with the cluster for Comprehend and S3 access |
| ✅ psycopg2 Connection | Connected to Aurora from Python (replacing pgAdmin) |
| ✅ AWS ML Extension | Installed `aws_ml` to enable `aws_comprehend` SQL functions |
| ✅ Sample Analysis | Ran `detect_sentiment()` on a `comments` table |
| ✅ S3 Import | Used `aws_s3.table_import_from_s3` to load the full TSV dataset |
| ✅ Bulk Scoring | Updated all review rows with Comprehend sentiment + confidence |
| ✅ Analytical Queries | Queried by sentiment category and confidence threshold |
| ✅ Cleanup | Deleted all resources to avoid ongoing charges |

---

### Key Learnings

- **Aurora ML** lets you call Amazon Comprehend (and SageMaker) directly from SQL without moving data or writing application code.
- **Caching sentiment in the DB** (storing results in columns) is far more cost-efficient than calling Comprehend on every query.
- **psycopg2** is a powerful Python client for PostgreSQL that makes Aurora fully accessible from SageMaker notebooks, scripts, or Lambda functions.
- **boto3** enables complete AWS infrastructure automation — IAM, RDS, EC2, and S3 — without needing the console.

---

### Documentation References

- [Amazon Aurora Overview](https://aws.amazon.com/rds/aurora/)
- [Aurora Machine Learning](https://aws.amazon.com/rds/aurora/machine-learning/)
- [Aurora ML SQL Functions](https://docs.aws.amazon.com/AmazonRDS/latest/AuroraUserGuide/aurora-ml.html)
- [Amazon Comprehend Pricing](https://aws.amazon.com/comprehend/pricing/)
- [AWS Tutorial: Sentiment Analysis with Aurora ML](https://aws.amazon.com/tutorials/sentiment-analysis-amazon-aurora-ml-integration/)